# 本ノートブックではNikkeiの情報から学会情報をまとめたDataFrameを抽出する

### 学会情報を集める。

In [1]:
import pandas as pd
import numpy as np
import datetime
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
#import lightgbm as lgb
import requests
from bs4 import BeautifulSoup
import time
import re
from urllib.request import Request, urlopen
#import optuna.integration.lightgbm as lgb_o
from itertools import combinations, permutations
import matplotlib.pyplot as plt


In [24]:
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

random.choice(USER_AGENTS)

'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107'

In [25]:
url = "https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202601.html"
headers = {'User-Agent': random.choice(USER_AGENTS)}
html = requests.get(url, headers=headers)
html.encoding = html.apparent_encoding 
# html.encoding = "EUC-JP"

In [15]:
soup = BeautifulSoup(html.text, "html.parser")

In [16]:
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="ja">
 <head prefix="og: http://ogp.me/ns# fb: http://ogp.me/ns/fb# website: http://ogp.me/ns/website#">
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <meta content="telephone=no" name="format-detection"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="on" http-equiv="x-dns-prefetch-control"/>
  <link href="//securepubads.g.doubleclick.net" rel="preconnect dns-prefetch"/>
  <link href="//www.googletagmanager.com" rel="preconnect dns-prefetch"/>
  <link href="//www.googletagservices.com" rel="preconnect dns-prefetch"/>
  <link href="//atlas-endpoint.bp.n8s.jp" rel="preconnect dns-prefetch"/>
  <link href="//adservice.google.co.jp" rel="preconnect dns-prefetch"/>
  <link href="//adservice.google.com" rel="preconnect dns-prefetch"/>
  <link href="//tpc.googlesyndication.com" rel="preconnect dns-prefetch"/>
  <link href="//pagead2.googlesyndication.com" rel="preconnect dns-

In [13]:
soup.find('h1')

<h1 class="logo"><a data-atlas-trackable="logo_nm" href="/inc/all/doctor/"><img alt="��ョ����＜����ｃ�������������雁�糸��" src="/images/logo/logo-doctor.png"/></a></h1>

In [17]:
data_list = []
base_url = "https://medical.nikkeibp.co.jp" # 相対パスを絶対パスにするためのベースURL

# すべての行（aタグ）を取得
rows = soup.find_all('a', class_='gakkai-list-row')

for row in rows:
    # タイトルを取得（空白除去）
    title_div = row.find('div', class_='gakkai-list-cell type-title')
    title = title_div.get_text(strip=True) if title_div else "不明"

    # URLを取得（href属性）
    link = row.get('href')
    # 相対パス（/inc/...）の場合はドメインを結合する
    full_url = base_url + link if link else ""
    
    # 日付と場所も一応取得しておくと便利です（不要なら削除可）
    date_div = row.find('div', class_='gakkai-list-cell type-date')
    date = date_div.get_text(strip=True) if date_div else ""
    
    place_div = row.find('div', class_='gakkai-list-cell type-place')
    place = place_div.get_text(strip=True) if place_div else ""

    # リストに追加
    data_list.append({
        '学会名': title,
        'URL': full_url,
        '開催日': date,
        '開催地': place
    })

# DataFrameに変換
df = pd.DataFrame(data_list)

# 結果の表示
print(df)

                                                 学会名  \
0                              JCRミッドウィンターセミナー（第39回）   
1                           日本成人病(生活習慣病)学会学術集会（第59回）   
2                                   日本糖尿病眼学会総会（第32回）   
3                                  大腸癌研究会学術集会（第104回）   
4                                  日本総合健診医学会大会（第54回）   
5                                    日本疫学会学術総会（第36回）   
6                               日本病態栄養学会年次学術集会（第29回）   
7  Asian Pacific Knee Osteotomy Symposium（The 8th. ）   
8                              日本性差医学・医療学会学術集会（第19回）   

                                                 URL                   開催日  \
0  https://medical.nikkeibp.co.jp/inc/all/gakkai/...   2026.01.10（土）-11（日）   
1  https://medical.nikkeibp.co.jp/inc/all/gakkai/...   2026.01.10（土）-11（日）   
2  https://medical.nikkeibp.co.jp/inc/all/gakkai/...   2026.01.16（金）-17（土）   
3  https://medical.nikkeibp.co.jp/inc/all/gakkai/...   2026.01.22（木）-23（金）   
4  https://medical.nikkeibp.co.jp/inc/all/gakkai/

In [20]:
df['URL'][0]

'https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/if2601001.html'

# 1年分の学会情報を抽出する

In [42]:
data_list = []

for month in range(1, 13):
    url = f"https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/2026{month:02}.html" # 0埋めが必要か確認
    # ※元のURL構造が int（1桁）で通るなら {month}、01等が必要なら {month:02}
    
    headers = {'User-Agent': random.choice(USER_AGENTS)}
    html = requests.get(url, headers=headers)
    html.encoding = html.apparent_encoding 
    soup = BeautifulSoup(html.text, "html.parser")
    
    print(f"Fetching: {url}")

    
    base_url = "https://medical.nikkeibp.co.jp" # 相対パスを絶対パスにするためのベースURL

    # すべての行（aタグ）を取得
    rows = soup.find_all('a', class_='gakkai-list-row')
    

    for row in rows:
        # タイトルを取得（空白除去）
        title_div = row.find('div', class_='gakkai-list-cell type-title')
        title = title_div.get_text(strip=True) if title_div else "不明"

        # URLを取得（href属性）
        link = row.get('href')
        # 相対パス（/inc/...）の場合はドメインを結合する
        full_url = base_url + link if link else ""
        
        # 日付と場所も一応取得しておくと便利です（不要なら削除可）
        date_div = row.find('div', class_='gakkai-list-cell type-date')
        date = date_div.get_text(strip=True) if date_div else ""
        
        place_div = row.find('div', class_='gakkai-list-cell type-place')
        place = place_div.get_text(strip=True) if place_div else ""

        # リストに追加
        data_list.append({
            '学会名': title,
            'URL': full_url,
            '開催日': date,
            '開催地': place
        })

# DataFrameに変換
df = pd.DataFrame(data_list)

Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202601.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202602.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202603.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202604.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202605.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202606.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202607.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202608.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202609.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202610.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202611.html
Fetching: https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/202612.html


In [43]:
df

,学会名,URL,開催日,開催地
0,JCRミッドウィンターセミナー（第39回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.01.10（土）-11（日）,京都市南区
1,日本成人病(生活習慣病)学会学術集会（第59回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.01.10（土）-11（日）,東京都千代田区
2,日本糖尿病眼学会総会（第32回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.01.16（金）-17（土）,横浜市西区
3,大腸癌研究会学術集会（第104回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.01.22（木）-23（金）,東京都港区
4,日本総合健診医学会大会（第54回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.01.23（金）-24（土）,横浜市西区
...,...,...,...,...
210,日本呼吸ケア・リハビリテーション学会学術集会（第36回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.11.27（金）-28（土）,横浜市西区
211,日本脳循環代謝学会学術集会（第69回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.11.27（金）-28（土）,福井市
212,日本小児アレルギー学会学術大会（第63回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.11.28（土）-29（日）,浜松市中区
213,日本形成外科学会基礎学術集会（第35回）,https://medical.nikkeibp.co.jp/inc/all/gakkai/...,2026.12.17（木）-18（金）,那覇市


In [44]:
# Excelで開くならこれが一番安全
# df.to_csv('2026国内学会.csv', encoding='utf-8-sig', index=False)

In [45]:
# 1. 月と日を抽出
df[['月', '日']] = df['開催日'].str.extract(r'2026\.(\d{1,2})\.(\d{1,2})')

# 2. 結果を確認
print(df[['開催日', '月', '日']].head())

                   開催日   月   日
0  2026.01.10（土）-11（日）  01  10
1  2026.01.10（土）-11（日）  01  10
2  2026.01.16（金）-17（土）  01  16
3  2026.01.22（木）-23（金）  01  22
4  2026.01.23（金）-24（土）  01  23


In [51]:
df[df['月']=='03']
df.iloc[47]['URL']

'https://medical.nikkeibp.co.jp/inc/all/gakkai/calendar/if2603012.html'